# 🐍➡️ Go: Python to Go Code Converter

This notebook demonstrates a powerful AI-powered tool for converting Python code to Go using multiple advanced LLM models. It helps developers quickly translate Python logic into idiomatic Go code while maintaining correctness and performance.

**Key Capabilities:**
- 🔄 Converts Python code to production-ready Go equivalents with high accuracy
- 🤖 Integrates multiple free AI models (OpenAI, DeepSeek, Groq) for comparison
- 📊 Scores and benchmarks conversion quality across different models
- 🖥️ Provides an interactive Gradio web interface for easy testing
- ⚡ Uses free accessible models (no expensive API costs)

Explore the cells below to understand Python-to-Go conversion mechanics, model comparison, and how AI helps bridge language paradigms.

### Step 1: Import Required Libraries
This cell imports the necessary libraries for the Python to Go code generator.

In [ ]:
# imports
import os
import re
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from groq import Groq
from groq import Groq

### Step 2: Load API Keys
This cell imports OpenRouter and Groq API keys from .env file

In [ ]:
load_dotenv()

# Check for API keys
openrouter_key = os.getenv("OPENROUTER_API_KEY")
groq_key = os.getenv("GROQ_API_KEY")

if not openrouter_key and not groq_key:
    raise EnvironmentError("Neither OPENROUTER_API_KEY nor GROQ_API_KEY found in .env file")

print("✅ Environment loaded successfully")
if openrouter_key:
    print("  - OpenRouter API key loaded")
if groq_key:
    print("  - Groq API key loaded")

✅ Environment loaded successfully
  - OpenRouter API key loaded
  - Groq API key loaded


### Step 3: Initialize API Clients
This cell initializes OpenRouter and Groq clients

In [18]:
# Initialize OpenRouter client
openrouter = None
if openrouter_key:
    openrouter = OpenAI(
        api_key=openrouter_key,
        base_url="https://openrouter.ai/api/v1"
    )
    print("✅ OpenRouter client initialized")

# Initialize Groq client
groq_client = None
if groq_key:
    groq_client = Groq(api_key=groq_key)
    print("✅ Groq client initialized")

✅ OpenRouter client initialized
✅ Groq client initialized


### Step 4: Prompt designing
This cell prepares appropriate system prompts 

In [19]:
SYSTEM_PROMPT = """
Convert the provided Python code into efficient, idiomatic, and production-quality Go.

Strict Requirements:
- Output ONLY valid Go code (no explanations, comments outside code, or markdown)
- The code MUST compile successfully without modification
- Include `package main` and a complete `func main()`
- Preserve the original logic, control flow, and behavior exactly
- Use only the Go standard library (no third-party packages)

Code Quality Constraints:
- Follow idiomatic Go conventions (naming, formatting, simplicity)
- Use explicit typing where appropriate (avoid unnecessary use of interface{})
- Handle errors properly using Go error patterns (do not ignore errors)
- Ensure correct variable initialization and zero-value handling
- Use appropriate Go data structures (slices, maps, structs) instead of direct Python equivalents

Behavioral Fidelity:
- Maintain equivalent input/output behavior
- Preserve edge case handling and boundary conditions
- Translate Python-specific constructs (e.g., list comprehensions, dynamic typing) into correct Go equivalents

Execution Safety:
- Ensure no runtime panics under normal conditions
- Avoid unused variables/imports
- Ensure all imports are necessary and used

Output Format:
- Return a single complete Go program
- No placeholders, pseudocode, or incomplete sections
"""

### Step 5: Models Listing

In [20]:
MODELS = {
    "OpenAI (GPT-4o Mini)": {"provider": "openrouter", "model": "openai/gpt-4o-mini"},
    "DeepSeek V3": {"provider": "openrouter", "model": "deepseek/deepseek-r1"},
    "Groq Mixtral 8x7b": {"provider": "groq", "model": "mixtral-8x7b-32768"},
    "Groq Llama 3 70b": {"provider": "groq", "model": "llama-3-70b-8192"}
}

print(f"✅ Loaded {len(MODELS)} models:")
for name, config in MODELS.items():
    print(f"  - {name} ({config['provider']})")

✅ Loaded 4 models:
  - OpenAI (GPT-4o Mini) (openrouter)
  - DeepSeek V3 (openrouter)
  - Groq Mixtral 8x7b (groq)
  - Groq Llama 3 70b (groq)


### Step 6: Function that converts python code to Go Lang

In [21]:
def convert_to_go(model_name, python_code):
    try:
        config = MODELS[model_name]
        provider = config["provider"]
        model_id = config["model"]
        
        if provider == "openrouter":
            if not openrouter:
                return None, "OpenRouter API key not configured"
            response = openrouter.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": python_code}
                ],
                temperature=0
            )
            return response.choices[0].message.content.strip(), None
        
        elif provider == "groq":
            if not groq_client:
                return None, "Groq API key not configured"
            response = groq_client.chat.completions.create(
                model=model_id,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": python_code}
                ],
                temperature=0
            )
            return response.choices[0].message.content.strip(), None
        
        else:
            return None, f"Unknown provider: {provider}"
    except Exception as e:
        return None, str(e)

### Step 7: This cell visualizes the results 

It benchmark the results and compares conversion accuracy.

In [22]:
def score_go_code(go_code, python_code):
    score = 0
    notes = []

    if "package main" in go_code:
        score += 2
        notes.append("✔ package main present")
    else:
        notes.append("✘ missing package main")

    if "func main()" in go_code:
        score += 2
        notes.append("✔ main function present")
    else:
        notes.append("✘ missing main function")

    if "fmt." in go_code:
        score += 1
        notes.append("✔ printing preserved")

    math_ops = ["*", "/", "+", "-"]
    for op in math_ops:
        if op in python_code and op in go_code:
            score += 1
            notes.append(f"✔ operator '{op}' preserved")

    py_vars = set(re.findall(r'\b[a-zA-Z_]\w*\b', python_code))
    go_vars = set(re.findall(r'\b[a-zA-Z_]\w*\b', go_code))
    shared = py_vars.intersection(go_vars)

    if shared:
        score += len(shared) * 0.3
        notes.append("✔ variables preserved: " + ", ".join(list(shared)[:5]))

    if "TODO" in go_code or "..." in go_code:
        score -= 3
        notes.append("✘ incomplete code detected")

    if "print(" in go_code:
        score -= 2
        notes.append("✘ Python print found")

    return score, notes

## Step 8: This function provides a user interface for interactive Python-to-Go code conversion using Gradio.

In [23]:
def evaluate_models(python_code, selected_models):
    results = []

    for name in selected_models:
        go_code, error = convert_to_go(name, python_code)

        if error:
            results.append({
                "model": name,
                "error": error
            })
            continue

        score, notes = score_go_code(go_code, python_code)

        results.append({
            "model": name,
            "go_code": go_code,
            "score": score,
            "notes": notes,
            "error": None
        })

    valid = [r for r in results if r["error"] is None]
    winner = max(valid, key=lambda x: x["score"]) if valid else None

    return results, winner

## Step 9: This function process the above functions as a gradio tool 

In [24]:
def process_code(python_code, selected_models):
    results, winner = evaluate_models(python_code, selected_models)

    output = ""

    for r in results:
        output += f"\nMODEL: {r['model']}\n"
        output += "-"*40 + "\n"

        if r["error"]:
            output += f"❌ Error:\n{r['error']}\n\n"
        else:
            output += r["go_code"] + "\n\n"
            output += f"Score: {r['score']:.2f}\n"
            output += "\n".join(r["notes"]) + "\n\n"

    if winner:
        output += f"\nBest model: {winner['model']} (Score: {winner['score']:.2f})\n"

    return output

## Step 10: Gradio Code that displays the implementation

In [26]:
with gr.Blocks(theme=gr.themes.Soft(), css="""
#title{
    text-align:center;
    font-size:34px;
    font-weight:bold;
    margin-bottom:10px;
}
#subtitle{
    text-align:center;
    font-size:16px;
    color:gray;
    margin-bottom:25px;
}
.gr-button{
    background: linear-gradient(to right, #4facfe, #00f2fe);
    color:white;
    border:none;
    font-size:16px;
    font-weight:bold;
    border-radius:10px;
}
""") as demo:

    gr.Markdown("<div id='title'>⚡ Python → Go AI Generator</div>")
    gr.Markdown("<div id='subtitle'>Compare Multiple LLMs and Automatically Pick Best Go Translation</div>")

    with gr.Row():

        with gr.Column(scale=1):
            python_input = gr.Code(
                label="🐍 Paste Python Code Here",
                language="python",
                lines=20
            )

            model_select = gr.CheckboxGroup(
                choices=list(MODELS.keys()),
                value=[list(MODELS.keys())[0]],
                label="🤖 Select AI Models"
            )

            run_btn = gr.Button("🚀 Generate Go Code")

        with gr.Column(scale=1):
            output_box = gr.Code(
                label="⚙ AI Conversion Results",
                language="python",
                lines=25
        )

    run_btn.click(
        fn=process_code,
        inputs=[python_input, model_select],
        outputs=output_box
    )

demo.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
